### Install Dependencies

In [ ]:
!pip install kagglehub
dbutils.library.restartPython()

### Define Parameters

Review the widgets before running. Defaults match the **Marketing Campaign Analytics** layout.

| Widget | Default | What it controls |
| --- | --- | --- |
| `CATALOG_NAME` | `workspace` | Unity Catalog for demo objects. Default `workspace` is pre-provisioned — no catalog creation needed. |
| `SCHEMA_NAME` | `marketing_campaign` | Schema for retail star-schema tables (`dim_*`, `fact_sales_normalized`). |


In [ ]:
dbutils.widgets.text("CATALOG_NAME", "workspace", "CATALOG_NAME — Unity Catalog catalog for demo objects")
dbutils.widgets.text("SCHEMA_NAME", "marketing_campaign", "SCHEMA_NAME — schema inside the catalog")
CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")

### Create Schema and `datasets` Volume

Uses the existing **`workspace`** catalog by default — only the schema and volume are created.


In [ ]:
if CATALOG_NAME.lower() != "workspace":
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.datasets")

DATASETS_DIR = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/datasets"
print("Datasets volume:", DATASETS_DIR)

### Download the Dataset from Kaggle and Copy the CSVs into `datasets`

`kagglehub` downloads into a local cache (nested by dataset/version). We copy the
CSV files **flat** into the `datasets` volume folder so the volume stays clean.

In [ ]:
import kagglehub, glob, os, shutil

# Download to the local kagglehub cache (nested folders live on local disk, not the volume)
download_path = kagglehub.dataset_download("shrinivasv/retail-store-star-schema-dataset")
print("Downloaded to:", download_path)

# Copy every CSV flat into the datasets volume
csv_files = glob.glob(f"{download_path}/**/*.csv", recursive=True)
for csv_file in csv_files:
    dst = os.path.join(DATASETS_DIR, os.path.basename(csv_file))
    shutil.copyfile(csv_file, dst)
    print("  copied", os.path.basename(csv_file))

print(f"\n{len(csv_files)} CSV files now in {DATASETS_DIR}")

### Load the CSV Files from `datasets` into Delta Tables

In [ ]:
import glob, os

for csv_file in glob.glob(f"{DATASETS_DIR}/*.csv"):
    table_name = os.path.splitext(os.path.basename(csv_file))[0]
    df = spark.read.csv(csv_file, header=True, inferSchema=True)
    # Clean column names: replace spaces with underscores
    for c in df.columns:
        df = df.withColumnRenamed(c, c.replace(" ", "_"))
    full_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{table_name}"
    spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")
    df.write.saveAsTable(full_table_name)
    print("  loaded", full_table_name)